# Vicilin Molecular Dynamics Pipeline**Protein:** Pea vicilin homodimer (UniProt Q702P1)**Pipeline:** AlphaFold3 structure → GROMACS MD simulation → trajectory analysis → PyMOL visualization**Conditions simulated:** 300K and 400K, each at 200ps and 500ps**Environment:** Google Colab (T4 GPU), GROMACS with CUDA support, AMBER99SB-ILDN force field, SPC/E water modelThis notebook reflects the final working pipeline. Exploratory/debugging cells andcross-checks against the legumin dataset have been removed for clarity — see therepository's `simulation_setup/vicilin/` folder for the exact `.mdp` parameter filesused at each stage.

## 1. Environment Setup

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os

In [ ]:
!pip install -q condacolabimport condacolabcondacolab.install()

In [ ]:
!mamba install -y -c conda-forge -c bioconda gromacs=2026.3=*cuda*

In [ ]:
%%bashgmx --version

## 2. System PreparationStarting from the AlphaFold3-predicted homodimer structure (`vicilin_dimer.pdb`,see `structures/` in the repository).

In [ ]:
%%bashcp "/content/drive/MyDrive/Vicilin_MD/vicilin_dimer.pdb" /content/run/cd /content/runls

In [ ]:
%%bashcd /content/rungmx pdb2gmx -f vicilin_dimer.pdb -o vicilin_processed.gro -water spce -ff amber99sb-ildn

In [ ]:
%%bashcd /content/rungmx editconf -f vicilin_processed.gro -o vicilin_box.gro -c -d 1.0 -bt cubic

In [ ]:
%%bashcd /content/rungmx solvate -cp vicilin_box.gro -cs spc216.gro -o vicilin_solv.gro -p topol.top

### 2a. Ion neutralization

In [ ]:
%%writefile /content/run/ions.mdp; ions.mdp - used as input into grompp to generate ions.tprintegrator  = steepemtol       = 1000.0emstep      = 0.01nsteps      = 50000nstlist         = 1cutoff-scheme  = Verletns_type         = gridcoulombtype     = cutoffrcoulomb        = 1.0rvdw            = 1.0pbc             = xyz

In [ ]:
%%bashcd /content/rungmx grompp -f ions.mdp -c vicilin_solv.gro -p topol.top -o ions.tpr

In [ ]:
%%bashcd /content/runecho 13 | gmx genion -s ions.tpr -o vicilin_ions.gro -p topol.top -pname NA -nname CL -neutral

In [ ]:
%%bashcp /content/run/vicilin_ions.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/topol.top /content/drive/MyDrive/Vicilin_MD/cp /content/run/ions.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/ions.mdp /content/drive/MyDrive/Vicilin_MD/cp /content/run/topol_Protein_chain_A.itp /content/drive/MyDrive/Vicilin_MD/cp /content/run/topol_Protein_chain_B.itp /content/drive/MyDrive/Vicilin_MD/

## 3. Energy Minimization

In [ ]:
%%writefile /content/run/em.mdp; em.mdp - energy minimizationintegrator  = steepemtol       = 1000.0emstep      = 0.01nsteps      = 50000nstlist         = 10cutoff-scheme   = Verletns_type         = gridcoulombtype     = PMErcoulomb        = 1.0rvdw            = 1.0pbc             = xyz

In [ ]:
%%bashcd /content/rungmx grompp -f em.mdp -c vicilin_ions.gro -p topol.top -o em.tpr

In [ ]:
%%bashcd /content/rungmx mdrun -deffnm em -nb gpu

In [ ]:
%%bashcp /content/run/em.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/em.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/em.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/em.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/em.mdp /content/drive/MyDrive/Vicilin_MD/

## 4. NVT and NPT Equilibration — 300K

In [ ]:
%%writefile /content/run/nvt.mdptitle       = NVT equilibrationdefine      = -DPOSRESintegrator  = mdnsteps      = 50000dt          = 0.002nstxout     = 500nstvout     = 500nstenergy   = 500nstlog      = 500continuation    = noconstraint_algorithm = lincsconstraints     = h-bondslincs_iter      = 1lincs_order     = 4cutoff-scheme   = Verletns_type         = gridnstlist         = 10rcoulomb        = 1.0rvdw            = 1.0coulombtype     = PMEpme_order       = 4fourierspacing  = 0.16tcoupl      = V-rescaletc-grps     = Protein Non-Proteintau_t       = 0.1     0.1ref_t       = 300     300pcoupl      = nopbc         = xyzDispCorr    = EnerPresgen_vel     = yesgen_temp    = 300gen_seed    = -1

In [ ]:
%%bashcd /content/rungmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr

In [ ]:
%%bashcd /content/rungmx mdrun -deffnm nvt -nb gpu -pme gpu -bonded gpu

In [ ]:
%%bashcp /content/run/nvt.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt.cpt /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt.mdp /content/drive/MyDrive/Vicilin_MD/

In [ ]:
%%writefile /content/run/npt.mdptitle       = NPT equilibrationdefine      = -DPOSRESintegrator  = mdnsteps      = 50000dt          = 0.002nstxout     = 500nstvout     = 500nstenergy   = 500nstlog      = 500continuation    = yesconstraint_algorithm = lincsconstraints     = h-bondslincs_iter      = 1lincs_order     = 4cutoff-scheme   = Verletns_type         = gridnstlist         = 10rcoulomb        = 1.0rvdw            = 1.0coulombtype     = PMEpme_order       = 4fourierspacing  = 0.16tcoupl      = V-rescaletc-grps     = Protein Non-Proteintau_t       = 0.1     0.1ref_t       = 300     300pcoupl          = C-rescalepcoupltype      = isotropictau_p           = 2.0ref_p           = 1.0compressibility = 4.5e-5pbc         = xyzDispCorr    = EnerPresgen_vel     = no

In [ ]:
%%bashcd /content/rungmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 1

In [ ]:
%%bashcd /content/rungmx mdrun -deffnm npt -nb gpu -pme gpu -bonded gpu

In [ ]:
%%bashcp /content/run/npt.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt.cpt /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt.mdp /content/drive/MyDrive/Vicilin_MD/

## 5. Production MD — 300K

### 5a. 200ps run

In [ ]:
%%writefile /content/run/md.mdptitle       = Production MD 300Kintegrator  = mdnsteps      = 100000dt          = 0.002nstxout     = 0nstvout     = 0nstfout     = 0nstenergy   = 1000nstlog      = 1000nstxout-compressed = 1000continuation    = yesconstraint_algorithm = lincsconstraints     = h-bondslincs_iter      = 1lincs_order     = 4cutoff-scheme   = Verletns_type         = gridnstlist         = 10rcoulomb        = 1.0rvdw            = 1.0coulombtype     = PMEpme_order       = 4fourierspacing  = 0.16tcoupl      = V-rescaletc-grps     = Protein Non-Proteintau_t       = 0.1     0.1ref_t       = 300     300pcoupl          = C-rescalepcoupltype      = isotropictau_p           = 2.0ref_p           = 1.0compressibility = 4.5e-5pbc         = xyzDispCorr    = EnerPresgen_vel     = no

In [ ]:
%%bashcd /content/rungmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr

In [ ]:
%%bashcd /content/rungmx mdrun -deffnm md -nb gpu -pme gpu -bonded gpu

In [ ]:
%%bashcp /content/run/md.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/md.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/md.xtc /content/drive/MyDrive/Vicilin_MD/cp /content/run/md.cpt /content/drive/MyDrive/Vicilin_MD/cp /content/run/md.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/md.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/md.mdp /content/drive/MyDrive/Vicilin_MD/

### 5b. Analysis — 300K/200ps

In [ ]:
%%bashcd /content/runecho 1 0 | gmx trjconv -s md.tpr -f md.xtc -o md_center.xtc -pbc mol -center

In [ ]:
%%bashcd /content/runecho 4 4 | gmx rms -s md.tpr -f md_center.xtc -o rmsd_300K.xvg -tu nsecho 1 | gmx gyrate -s md.tpr -f md_center.xtc -o gyrate_300K.xvgecho 1 | gmx rmsf -s md.tpr -f md_center.xtc -o rmsf_300K.xvg -resecho 1 | gmx sasa -s md.tpr -f md_center.xtc -o sasa_300K.xvggmx hbond -s md.tpr -f md_center.xtc -num hbnum_300K.xvg -r "Protein" -t "Protein"

Split chain-A/chain-B index groups for per-chain RMSF:

In [ ]:
%%bashcd /content/rungmx make_ndx -f md.tpr -o chains.ndx << EOFsplitch 1qEOF

In [ ]:
%%bashcd /content/runecho "Protein_chain1" | gmx rmsf -s md.tpr -f md_center.xtc -n chains.ndx -o rmsf_chainA_300K.xvg -resecho "Protein_chain2" | gmx rmsf -s md.tpr -f md_center.xtc -n chains.ndx -o rmsf_chainB_300K.xvg -res

In [ ]:
%%bashcp /content/run/*.xvg /content/drive/MyDrive/Vicilin_MD/

### 5c. Extend to 500ps

In [ ]:
%%bashcd /content/rungmx convert-tpr -s md.tpr -extend 300 -o md_500ps.tpr

In [ ]:
%%bashcd /content/rungmx mdrun -deffnm md_500ps -s md_500ps.tpr -cpi md.cpt -nb gpu -pme gpu -bonded gpu -noappend

In [ ]:
%%bashcd /content/rungmx trjcat -f md.xtc md_500ps.part0002.xtc -o md_500ps_full.xtc

In [ ]:
%%bashcp /content/run/md_500ps.part0002.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_500ps.part0002.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_500ps.part0002.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_500ps.part0002.xtc /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_500ps.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_500ps_full.xtc /content/drive/MyDrive/Vicilin_MD/

### 5d. Analysis — 300K/500ps

In [ ]:
%%bashcd /content/runecho 1 0 | gmx trjconv -s md_500ps.tpr -f md_500ps_full.xtc -o md_500ps_center.xtc -pbc mol -center

In [ ]:
%%bashcd /content/runecho 4 4 | gmx rms -s md_500ps.tpr -f md_500ps_center.xtc -o rmsd_500ps.xvg -tu psecho 1 | gmx gyrate -s md_500ps.tpr -f md_500ps_center.xtc -o gyrate_500ps.xvgecho "Protein_chain1" | gmx rmsf -s md_500ps.tpr -f md_500ps_center.xtc -n chains.ndx -o rmsf_chainA_500ps.xvg -resecho "Protein_chain2" | gmx rmsf -s md_500ps.tpr -f md_500ps_center.xtc -n chains.ndx -o rmsf_chainB_500ps.xvg -resecho 1 | gmx sasa -s md_500ps.tpr -f md_500ps_center.xtc -o sasa_500ps.xvggmx hbond -s md_500ps.tpr -f md_500ps_center.xtc -num hbnum_500ps.xvg -r "Protein" -t "Protein"

In [ ]:
%%bashcp /content/run/rmsd_500ps.xvg /content/drive/MyDrive/Vicilin_MD/cp /content/run/gyrate_500ps.xvg /content/drive/MyDrive/Vicilin_MD/cp /content/run/rmsf_chainA_500ps.xvg /content/drive/MyDrive/Vicilin_MD/cp /content/run/rmsf_chainB_500ps.xvg /content/drive/MyDrive/Vicilin_MD/cp /content/run/sasa_500ps.xvg /content/drive/MyDrive/Vicilin_MD/cp /content/run/hbnum_500ps.xvg /content/drive/MyDrive/Vicilin_MD/

## 6. NVT, NPT, and Production MD — 400K

Same procedure as 300K, repeated with the 400K parameter set.

In [ ]:
%%writefile /content/run/nvt_400K.mdptitle       = NVT equilibration 400Kdefine      = -DPOSRESintegrator  = mdnsteps      = 50000dt          = 0.002nstxout     = 500nstvout     = 500nstenergy   = 500nstlog      = 500continuation    = noconstraint_algorithm = lincsconstraints     = h-bondslincs_iter      = 1lincs_order     = 4cutoff-scheme   = Verletns_type         = gridnstlist         = 10rcoulomb        = 1.0rvdw            = 1.0coulombtype     = PMEpme_order       = 4fourierspacing  = 0.16tcoupl      = V-rescaletc-grps     = Protein Non-Proteintau_t       = 0.1     0.1ref_t       = 400     400pcoupl      = nopbc         = xyzDispCorr    = EnerPresgen_vel     = yesgen_temp    = 400gen_seed    = -1

In [ ]:
%%bashcd /content/rungmx grompp -f nvt_400K.mdp -c em.gro -r em.gro -p topol.top -o nvt_400K.tprgmx mdrun -deffnm nvt_400K -nb gpu -pme gpu -bonded gpu

In [ ]:
%%bashcp /content/run/nvt_400K.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt_400K.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt_400K.cpt /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt_400K.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt_400K.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/nvt_400K.mdp /content/drive/MyDrive/Vicilin_MD/

In [ ]:
%%writefile /content/run/npt_400K.mdptitle       = NPT equilibration 400Kdefine      = -DPOSRESintegrator  = mdnsteps      = 50000dt          = 0.002nstxout     = 500nstvout     = 500nstenergy   = 500nstlog      = 500continuation    = yesconstraint_algorithm = lincsconstraints     = h-bondslincs_iter      = 1lincs_order     = 4cutoff-scheme   = Verletns_type         = gridnstlist         = 10rcoulomb        = 1.0rvdw            = 1.0coulombtype     = PMEpme_order       = 4fourierspacing  = 0.16tcoupl      = V-rescaletc-grps     = Protein Non-Proteintau_t       = 0.1     0.1ref_t       = 400     400pcoupl          = C-rescalepcoupltype      = isotropictau_p           = 2.0ref_p           = 1.0compressibility = 4.5e-5pbc         = xyzDispCorr    = EnerPresgen_vel     = no

In [ ]:
%%bashcd /content/rungmx grompp -f npt_400K.mdp -c nvt_400K.gro -r nvt_400K.gro -t nvt_400K.cpt -p topol.top -o npt_400K.tpr -maxwarn 1gmx mdrun -deffnm npt_400K -nb gpu -pme gpu -bonded gpu

In [ ]:
%%bashcp /content/run/npt_400K.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt_400K.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt_400K.cpt /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt_400K.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt_400K.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/npt_400K.mdp /content/drive/MyDrive/Vicilin_MD/

### 6a. 200ps production

In [ ]:
%%writefile /content/run/md_400K.mdptitle       = Production MD 400Kintegrator  = mdnsteps      = 100000dt          = 0.002nstxout     = 0nstvout     = 0nstfout     = 0nstenergy   = 1000nstlog      = 1000nstxout-compressed = 1000continuation    = yesconstraint_algorithm = lincsconstraints     = h-bondslincs_iter      = 1lincs_order     = 4cutoff-scheme   = Verletns_type         = gridnstlist         = 10rcoulomb        = 1.0rvdw            = 1.0coulombtype     = PMEpme_order       = 4fourierspacing  = 0.16tcoupl      = V-rescaletc-grps     = Protein Non-Proteintau_t       = 0.1     0.1ref_t       = 400     400pcoupl          = C-rescalepcoupltype      = isotropictau_p           = 2.0ref_p           = 1.0compressibility = 4.5e-5pbc         = xyzDispCorr    = EnerPresgen_vel     = no

In [ ]:
%%bashcd /content/rungmx grompp -f md_400K.mdp -c npt_400K.gro -t npt_400K.cpt -p topol.top -o md_400K.tprgmx mdrun -deffnm md_400K -nb gpu -pme gpu -bonded gpu

In [ ]:
%%bashcp /content/run/md_400K.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K.xtc /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K.cpt /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K.mdp /content/drive/MyDrive/Vicilin_MD/

In [ ]:
%%bashcd /content/runecho 1 0 | gmx trjconv -s md_400K.tpr -f md_400K.xtc -o md_400K_center.xtc -pbc mol -center

In [ ]:
%%bashcd /content/runecho 4 4 | gmx rms -s md_400K.tpr -f md_400K_center.xtc -o rmsd_400K_200ps.xvg -tu psecho 1 | gmx gyrate -s md_400K.tpr -f md_400K_center.xtc -o gyrate_400K_200ps.xvgecho "Protein_chain1" | gmx rmsf -s md_400K.tpr -f md_400K_center.xtc -n chains.ndx -o rmsf_chainA_400K_200ps.xvg -resecho "Protein_chain2" | gmx rmsf -s md_400K.tpr -f md_400K_center.xtc -n chains.ndx -o rmsf_chainB_400K_200ps.xvg -resecho 1 | gmx sasa -s md_400K.tpr -f md_400K_center.xtc -o sasa_400K_200ps.xvggmx hbond -s md_400K.tpr -f md_400K_center.xtc -num hbnum_400K_200ps.xvg -r "Protein" -t "Protein"

In [ ]:
%%bashcp /content/run/*_400K_200ps.xvg /content/drive/MyDrive/Vicilin_MD/

### 6b. Extend to 500ps

In [ ]:
%%bashcd /content/rungmx convert-tpr -s md_400K.tpr -extend 300 -o md_400K_500ps.tpr

In [ ]:
%%bashcd /content/rungmx mdrun -deffnm md_400K_500ps -s md_400K_500ps.tpr -cpi md_400K.cpt -nb gpu -pme gpu -bonded gpu -noappend

In [ ]:
%%bashcd /content/rungmx trjcat -f md_400K.xtc md_400K_500ps.part0002.xtc -o md_400K_500ps_full.xtc

In [ ]:
%%bashcp /content/run/md_400K_500ps.part0002.gro /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K_500ps.part0002.log /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K_500ps.part0002.edr /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K_500ps.part0002.xtc /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K_500ps.tpr /content/drive/MyDrive/Vicilin_MD/cp /content/run/md_400K_500ps_full.xtc /content/drive/MyDrive/Vicilin_MD/

In [ ]:
%%bashcd /content/runecho 1 0 | gmx trjconv -s md_400K_500ps.tpr -f md_400K_500ps_full.xtc -o md_400K_500ps_center.xtc -pbc mol -center

In [ ]:
%%bashcd /content/runecho 4 4 | gmx rms -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -o rmsd_400K_500ps.xvg -tu psecho 1 | gmx gyrate -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -o gyrate_400K_500ps.xvgecho "Protein_chain1" | gmx rmsf -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -n chains.ndx -o rmsf_chainA_400K_500ps.xvg -resecho "Protein_chain2" | gmx rmsf -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -n chains.ndx -o rmsf_chainB_400K_500ps.xvg -resecho 1 | gmx sasa -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -o sasa_400K_500ps.xvggmx hbond -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -num hbnum_400K_500ps.xvg -r "Protein" -t "Protein"

In [ ]:
%%bashcp /content/run/*_400K_500ps.xvg /content/drive/MyDrive/Vicilin_MD/

## 7. PlottingExample shown for one condition (400K/500ps) — the same `load_xvg` + matplotlibpattern was applied to all four conditions to produce the per-protein plots in`analysis/vicilin/`.

In [ ]:
import osimport numpy as npimport matplotlib.pyplot as pltdef load_xvg(path):    data = []    with open(path) as f:        for line in f:            if line.startswith(('#', '@')):                continue            data.append([float(x) for x in line.split()])    return np.array(data)run_dir = '/content/run'graphs_dir = '/content/drive/MyDrive/Vicilin_MD/Vicilin_400K_500ps'os.makedirs(graphs_dir, exist_ok=True)# RMSDrmsd = load_xvg(f'{run_dir}/rmsd_400K_500ps.xvg')plt.figure(figsize=(7, 4.5))plt.plot(rmsd[:,0], rmsd[:,1], color='blue', linewidth=1.2)plt.title('Vicilin Dimer Backbone RMSD — 400K (500 ps)')plt.xlabel('Time (ps)')plt.ylabel('RMSD (nm)')plt.tight_layout()plt.savefig(f'{graphs_dir}/rmsd_400K_500ps.png', dpi=300)plt.show()# Radius of gyrationrg = load_xvg(f'{run_dir}/gyrate_400K_500ps.xvg')plt.figure(figsize=(7, 4.5))plt.plot(rg[:,0], rg[:,1], color='blue', linewidth=1.2)plt.title('Radius of Gyration — 400K (500 ps)')plt.xlabel('Time (ps)')plt.ylabel('Rg (nm)')plt.tight_layout()plt.savefig(f'{graphs_dir}/rg_400K_500ps.png', dpi=300)plt.show()# RMSF (both chains)rmsf_a = load_xvg(f'{run_dir}/rmsf_chainA_400K_500ps.xvg')rmsf_b = load_xvg(f'{run_dir}/rmsf_chainB_400K_500ps.xvg')plt.figure(figsize=(10, 5))plt.plot(rmsf_a[:,0], rmsf_a[:,1], color='blue', linewidth=1.3, label='Chain A')plt.plot(rmsf_b[:,0], rmsf_b[:,1], color='red', linewidth=1.3, linestyle='--', label='Chain B')plt.title('RMSF of Vicilin at 400K (500 ps)')plt.xlabel('Residue')plt.ylabel('RMSF (nm)')plt.legend()plt.tight_layout()plt.savefig(f'{graphs_dir}/rmsf_400K_500ps.png', dpi=300)plt.show()# SASAsasa = load_xvg(f'{run_dir}/sasa_400K_500ps.xvg')plt.figure(figsize=(7, 4.5))plt.plot(sasa[:,0], sasa[:,1], color='blue', linewidth=1.2)plt.title('SASA — 400K Production (500 ps)')plt.xlabel('Time (ps)')plt.ylabel('SASA (nm²)')plt.tight_layout()plt.savefig(f'{graphs_dir}/sasa_400K_500ps.png', dpi=300)plt.show()# H-bondshbond = load_xvg(f'{run_dir}/hbnum_400K_500ps.xvg')plt.figure(figsize=(7, 4.5))plt.plot(hbond[:,0], hbond[:,1], color='blue', linewidth=1.2)plt.title('Hydrogen Bonds — 400K Production (500 ps)')plt.xlabel('Time (ps)')plt.ylabel('Number of H-bonds')plt.tight_layout()plt.savefig(f'{graphs_dir}/hbond_400K_500ps.png', dpi=300)plt.show()print("All graphs saved to:", graphs_dir)

### RMSF duration comparison (200ps vs 500ps)Both chains, overlaid across durations — used to produce`analysis/vicilin/rmsf_400K_200ps_vs_500ps.png` and the 300K equivalent.

In [ ]:
rmsf_a_200 = load_xvg(f'{run_dir}/rmsf_chainA_400K_200ps.xvg')rmsf_b_200 = load_xvg(f'{run_dir}/rmsf_chainB_400K_200ps.xvg')rmsf_a_500 = load_xvg(f'{run_dir}/rmsf_chainA_400K_500ps.xvg')rmsf_b_500 = load_xvg(f'{run_dir}/rmsf_chainB_400K_500ps.xvg')plt.figure(figsize=(10, 5))plt.plot(rmsf_a_200[:,0], rmsf_a_200[:,1], color='lightblue', linewidth=1.3, label='Chain A — 200 ps')plt.plot(rmsf_b_200[:,0], rmsf_b_200[:,1], color='lightblue', linewidth=1.3, linestyle=':', label='Chain B — 200 ps')plt.plot(rmsf_a_500[:,0], rmsf_a_500[:,1], color='red', linewidth=1.3, label='Chain A — 500 ps')plt.plot(rmsf_b_500[:,0], rmsf_b_500[:,1], color='red', linewidth=1.3, linestyle=':', label='Chain B — 500 ps')plt.title('RMSF Comparison of Vicilin at 400K — 200 ps vs 500 ps')plt.xlabel('Residue')plt.ylabel('RMSF (nm)')plt.legend()plt.tight_layout()plt.savefig(f'{graphs_dir}/rmsf_400K_200ps_vs_500ps.png', dpi=300)plt.show()

## 8. Trajectory Visualization (PyMOL)Renders the trajectory as a colored cartoon (chain A sky blue, chain B salmon),assembled into a GIF/MP4 with `ffmpeg`.

In [ ]:
%%bashpip install pymol-open-source --break-system-packages -q

In [ ]:
%%bashcp "/content/drive/MyDrive/Vicilin_MD/md_500ps.part0002.gro" /content/run/cp "/content/drive/MyDrive/Vicilin_MD/md_300K_500ps_center.xtc" /content/run/

In [ ]:
import pymolfrom pymol import cmdpymol.finish_launching(['pymol', '-qc'])cmd.load('/content/run/md_500ps.part0002.gro', 'vicilin300K')cmd.load_traj('/content/run/md_300K_500ps_center.xtc', 'vicilin300K', format='xtc')cmd.bg_color('white')cmd.hide('everything')cmd.show('cartoon')cmd.color('skyblue', 'chain A')cmd.color('salmon', 'chain B')cmd.zoom('all', -28)import osos.makedirs('/content/run/vicilin300K_frames', exist_ok=True)n = cmd.count_states('vicilin300K')print(f"Total frames: {n}")for i in range(1, n + 1):    cmd.frame(i)    cmd.png(f'/content/run/vicilin300K_frames/frame_{i:04d}.png', width=1280, height=960, dpi=150, ray=1)

In [ ]:
%%bashcd /content/run/vicilin300K_framesffmpeg -framerate 24 -i frame_%04d.png -vf "fps=24,scale=1200:-1:flags=lanczos" -pix_fmt yuv420p vicilin300K_500ps.mp4

In [ ]:
%%bashcp /content/run/vicilin300K_frames/vicilin300K_500ps.mp4 /content/drive/MyDrive/Vicilin_MD/